# Notebook 07 — Validation finale d'intégration (Phase 8, Rebuild 2026)

| | |
|---|---|
| **Projet** | Application Intelligente de Scoring Client |
| **Entreprise** | Orus Services — filiale Salafin, Bank of Africa |
| **Auteure** | Fatima Zahra Ait Lamine |
| **Année** | PFE 2025–2026 (rebuild du pipeline ML) |

---

## Objectif

Validation finale avant remise :

1. **Complétude du dépôt** : chaque notebook, artefact, figure, rapport et fichier de
   production attendu est présent.
2. **Intégration Spring Boot ↔ ml-service au niveau du contrat** : le payload est construit
   exactement comme `IaService.buildPayload` (montants MAD, conversion USD, ratio
   d'utilisation) et la réponse est consommée exactement comme `IaService` le fait
   (`score`, `narration`, `resoudreNiveau`, `mapExplications`, règle d'endettement aval).
   *(Le test full-stack avec PostgreSQL + Maven est documenté en procédure manuelle dans
   `docs/phase8_final_validation.md` — il requiert Docker et le JDK.)*
3. **Mapping à trois niveaux** : seuil de décision approuvé (0.22265625, inchangé) +
   seuil de surveillance F2-optimal (0.1007) — composition des bandes sur OOF.
4. **Explication du seuil de décision** : critère, protocole, voisinage, sélection automatique.
5. **Démarrage en session vierge** + synthèse de déploiement.

## 1. Complétude du dépôt (notebooks, artefacts, figures, rapports, production)

In [1]:
import sys, os, pickle, subprocess, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))        # ml-service/ : main.py
sys.path.append(os.path.abspath('../src'))    # module preprocessing

ATTENDUS = {
    'notebooks': [f'{n}.ipynb' for n in ['01_EDA', '02_preprocessing', '03_training',
                                          '04_evaluation', '05_export', '06_explainability',
                                          '07_integration_validation']],
    'figures': ['fig_R01_class_imbalance.png', 'fig_R01_default_rates_by_band.png',
                'fig_R02_monotonic_verification.png',
                'fig_R03_calibration_curves.png', 'fig_R03_roc_pr_curves.png',
                'fig_R03_threshold_optimization.png',
                'fig_R04_confusion_matrix.png', 'fig_R04_prob_distribution.png',
                'fig_R04_reliability.png', 'fig_R04_roc_pr.png',
                'fig_R06_shap_bar.png', 'fig_R06_shap_beeswarm.png',
                'fig_R06_waterfall_faible.png', 'fig_R06_waterfall_intermediaire.png',
                'fig_R06_waterfall_eleve.png', 'fig_R06_dependence.png'],
    'models': ['model_final.pkl', 'calibrator.pkl', 'decision_threshold.pkl',
               'niveau_moyen_threshold.pkl', 'feature_cols.pkl', 'metadata_final.pkl',
               'preprocessor.pkl', 'monotone_constraints.pkl', 'sample_check.pkl',
               'model_random_forest.pkl', 'calibrator_random_forest.pkl',
               'model_logistic_regression.pkl', 'calibrator_logistic_regression.pkl',
               'cv_results_phase3.csv', 'calibration_comparison_phase3.csv',
               'oof_predictions.csv', 'test_metrics_phase4.csv',
               'calibration_table_test_phase4.csv'],
    'docs': ['phase1_dataset_audit.md', 'phase2_preprocessing.md',
             'phase3_model_development.md', 'phase4_final_evaluation.md',
             'phase5_export.md', 'phase6_explainability.md',
             'phase7_application_integration.md'],
    'production': ['../main.py', '../src/preprocessing.py', '../tests/test_api_e2e.py',
                   '../README.md', '../requirements.txt', '../requirements-notebooks.txt'],
}
BASES = {'notebooks': '.', 'figures': 'figures', 'models': '../models', 'docs': '../docs', 'production': '.'}

manquants = []
for cat, fichiers in ATTENDUS.items():
    for f in fichiers:
        p = os.path.join(BASES[cat], f)
        if not os.path.exists(p) or os.path.getsize(p) == 0:
            manquants.append(p)
    print(f"  {cat:<12} : {len(fichiers):>2} fichiers attendus — "
          f"{'tous présents ✓' if not any(m.startswith(BASES[cat]) or cat=='production' and m in [os.path.join(BASES[cat], x) for x in fichiers] for m in manquants) else 'MANQUANTS !'}")
assert not manquants, f"fichiers manquants : {manquants}"
total = sum(len(v) for v in ATTENDUS.values())
print(f"\n✓ {total} fichiers vérifiés, 0 manquant (docs/phase8_final_validation.md est produit après ce notebook)")

  notebooks    :  7 fichiers attendus — tous présents ✓
  figures      : 16 fichiers attendus — tous présents ✓
  models       : 18 fichiers attendus — tous présents ✓
  docs         :  7 fichiers attendus — tous présents ✓
  production   :  6 fichiers attendus — tous présents ✓

✓ 54 fichiers vérifiés, 0 manquant (docs/phase8_final_validation.md est produit après ce notebook)


## 2. Intégration Spring Boot ↔ ml-service (niveau contrat)

On reproduit fidèlement les deux côtés de l'intégration :
- **Côté envoi** : `IaService.buildPayload` — client Orus en MAD, conversion USD (taux 10),
  `DebtRatio = charges/revenus`, utilisation % → ratio, champs composites calculés.
- **Côté consommation** : lecture de `score`/`narration`, `resoudreNiveau` (repli numérique
  si niveau invalide), `mapExplications` (sous-champs obligatoires), puis la règle métier
  aval `appliquerRegleEndettement` (taux ≥ 50 % ⇒ plancher MOYEN).

In [2]:
import main                     # démarre l'app : charge + valide les artefacts
from fastapi.testclient import TestClient
client_http = TestClient(main.app)

def build_payload_comme_iaservice(revenus_mad, charges_mad, age, r30, r60, r90,
                                  credits_ouverts, prets_immo, util_pct, dependents,
                                  historique_num=0.0, taux_mad_usd=10.0):
    """Réplique exacte de IaService.buildPayload (montants MAD côté backend)."""
    revenus_usd = revenus_mad / taux_mad_usd if taux_mad_usd > 0 else revenus_mad
    debt_ratio = charges_mad / revenus_mad if revenus_mad > 0 else 0.0
    return {
        "RevolvingUtilizationOfUnsecuredLines": util_pct / 100.0,
        "age": float(age),
        "NumberOfTime30-59DaysPastDueNotWorse": float(r30),
        "DebtRatio": debt_ratio,
        "MonthlyIncome": revenus_usd,
        "NumberOfOpenCreditLinesAndLoans": float(credits_ouverts),
        "NumberOfTimes90DaysLate": float(r90),
        "NumberRealEstateLoansOrLines": float(prets_immo),
        "NumberOfTime60-89DaysPastDueNotWorse": float(r60),
        "NumberOfDependents": float(dependents),
        "charges_mensuelles": debt_ratio * revenus_usd,
        "score_retards": r30 * 1.0 + r60 * 2.0 + r90 * 3.0,
        "historique_financier": historique_num,
        "nb_credits_total": float(credits_ouverts + prets_immo),
    }

def consommer_comme_iaservice(result, taux_endettement_pct):
    """Réplique de la consommation IaService : champs lus + règle d'endettement aval."""
    valeur_score = float(result["score"])                       # ((Number) result.get("score"))
    narration = str(result["narration"])
    niveau = str(result["niveau_risque"]).strip().upper()       # resoudreNiveau (chemin nominal)
    assert niveau in ("FAIBLE", "MOYEN", "ELEVE"), f"NiveauRisque.valueOf échouerait : {niveau}"
    explications = []
    for f in result["facteurs"]:                                # mapExplications
        explications.append({
            "featureName": str(f["feature_name"]),
            "shapValue": float(f["shap_value"]),
            "direction": bool(f["direction"]),
            "ordreImportance": int(f["ordre_importance"]),
        })
    if taux_endettement_pct is not None and taux_endettement_pct >= 50.0:   # appliquerRegleEndettement
        niveau = "MOYEN" if niveau == "FAIBLE" else niveau
        narration += "\n\n⚠ Taux d'endettement élevé..."
    return {"valeurScore": valeur_score, "niveauRisque": niveau,
            "narration": narration, "explications": explications}

# Client Orus réaliste : 15 000 MAD de revenus, 5 250 MAD de charges (35 %), 42 ans,
# 1 retard 30-59j, utilisation 40 %, 5 crédits + 1 prêt immo, 2 personnes à charge
payload = build_payload_comme_iaservice(15000, 5250, 42, 1, 0, 0, 5, 1, 40.0, 2)
rep = client_http.post("/predict", json=payload)
assert rep.status_code == 200, rep.text
entite_score = consommer_comme_iaservice(rep.json(), taux_endettement_pct=35.0)

print("Payload backend (extrait) : MonthlyIncome (USD) =", payload["MonthlyIncome"],
      "| DebtRatio =", payload["DebtRatio"])
print()
print("Entité Score telle que persistée par le backend (table `scores` + `explications`) :")
print(f"  valeurScore  : {entite_score['valeurScore']}  (PD calibrée × 100)")
print(f"  niveauRisque : {entite_score['niveauRisque']}")
print(f"  explications : {len(entite_score['explications'])} facteurs — "
      f"{[e['featureName'] for e in entite_score['explications']]}")
print(f"  narration    : {entite_score['narration'][:105]}...")

# Règle aval : un dossier FAIBLE avec taux d'endettement 55 % doit être plancher MOYEN
payload_endette = build_payload_comme_iaservice(20000, 11000, 50, 0, 0, 0, 6, 1, 10.0, 1)
rep2 = client_http.post("/predict", json=payload_endette).json()
aval = consommer_comme_iaservice(rep2, taux_endettement_pct=55.0)
assert rep2["niveau_risque"] == "FAIBLE" and aval["niveauRisque"] == "MOYEN"
print(f"\n✓ Règle métier aval vérifiée : ML=FAIBLE (PD {rep2['probabilite_calibree']*100:.1f}%) "
      f"+ endettement 55% → persisté MOYEN (escalade backend inchangée)")
print("✓ Tous les champs lus par IaService sont présents et typés correctement.")

2026-07-06 11:34:42,696 INFO Chargement des artefacts de production...


2026-07-06 11:34:43,220 INFO ✓ Modèle : XGBoost (2.0-rebuild-2026-07) — calibration isotonic, seuil 0.2227


2026-07-06 11:34:43,222 INFO   ROC-AUC test officiel : 0.8627


2026-07-06 11:34:43,223 INFO   Features (12) : ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'delinq_info_missing', 'income_missing']


2026-07-06 11:34:43,428 INFO Prédiction — age=42 : PD brute=0.5427 → calibrée=0.0885, niveau=FAIBLE, ACCEPTE


2026-07-06 11:34:43,440 INFO HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


2026-07-06 11:34:43,497 INFO Prédiction — age=50 : PD brute=0.1820 → calibrée=0.0197, niveau=FAIBLE, ACCEPTE


2026-07-06 11:34:43,505 INFO HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


Payload backend (extrait) : MonthlyIncome (USD) = 1500.0 | DebtRatio = 0.35

Entité Score telle que persistée par le backend (table `scores` + `explications`) :
  valeurScore  : 8.8  (PD calibrée × 100)
  niveauRisque : FAIBLE
  explications : 3 facteurs — ['NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTimes90DaysLate', 'NumberOfOpenCreditLinesAndLoans']
  narration    : Ce client de 42 ans présente un niveau de risque faible (probabilité de défaut estimée : 8.8 %, score 9/1...

✓ Règle métier aval vérifiée : ML=FAIBLE (PD 2.0%) + endettement 55% → persisté MOYEN (escalade backend inchangée)
✓ Tous les champs lus par IaService sont présents et typés correctement.


## 3. Mapping à trois niveaux — seuils finaux (composition sur OOF, train uniquement)

- **Seuil de décision (ÉLEVÉ)** : `decision_threshold.pkl` = **0.22265625** — inchangé,
  F1-optimal (phase 3), généralisation validée sur le test (phase 4).
- **Seuil de surveillance (MOYEN)** : `niveau_moyen_threshold.pkl` = **0.1007** —
  F2-optimal sur les mêmes prédictions OOF calibrées (rappel pondéré 2×). Rôle :
  **watch-list uniquement** — ce seuil n'affecte ni le modèle entraîné, ni la calibration,
  ni le seuil de décision approuvé ; il ne sert qu'à la catégorisation affichée
  FAIBLE/MOYEN/ÉLEVÉ. Stabilité inter-folds : 0.1008 ± 0.0104.

In [3]:
# ── Génération CANONIQUE du seuil de surveillance (source de vérité du dépôt) ──
# Entrée UNIQUE : les prédictions OOF calibrées de la phase 3 (train uniquement).
# Critère : F2 = (1+β²)·P·R / (β²·P + R) avec β = 2 — le rappel pèse deux fois plus
# que la précision : au stade de la watch-list, manquer un futur défaut coûte une
# perte de crédit potentielle, une entrée à tort ne coûte qu'un suivi.
import hashlib, pickle
def md5(path):
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()
INTOUCHABLES = ['../models/model_final.pkl', '../models/calibrator.pkl',
                '../models/decision_threshold.pkl']
hashes_avant = {f: md5(f) for f in INTOUCHABLES}

from sklearn.metrics import precision_recall_curve
oof = pd.read_csv('../models/oof_predictions.csv')
y, p = oof['y'].values, oof['p_prod_cal'].values

prec, rec, thr = precision_recall_curve(y, p)          # énumère toutes les valeurs prédites distinctes
BETA = 2.0
fbeta = (1 + BETA**2) * prec * rec / np.clip(BETA**2 * prec + rec, 1e-12, None)
t_surveillance = float(thr[int(np.nanargmax(fbeta[:-1]))])
with open('../models/niveau_moyen_threshold.pkl', 'wb') as f:
    pickle.dump(t_surveillance, f, protocol=4)

# Stabilité inter-folds (mêmes folds que la phase 3, seed 42)
from sklearn.model_selection import StratifiedKFold
def t_f2(yy, pp):
    pr, rc, th = precision_recall_curve(yy, pp)
    fb = 5 * pr * rc / np.clip(4 * pr + rc, 1e-12, None)
    return float(th[int(np.nanargmax(fb[:-1]))])
folds = [t_f2(y[va], p[va]) for _, va in
         StratifiedKFold(5, shuffle=True, random_state=42).split(p.reshape(-1, 1), y)]

# Indépendance : le calcul n'a lu QUE oof_predictions.csv ; les artefacts du modèle,
# du calibreur et du seuil de décision sont inchangés au bit près.
for f in INTOUCHABLES:
    assert md5(f) == hashes_avant[f], f'{f} modifié !'
assert abs(t_surveillance - main.seuil_surveillance) < 1e-15   # = valeur chargée par le service

print(f"niveau_moyen_threshold.pkl = {t_surveillance!r}")
print(f"Stabilité inter-folds : {np.mean(folds):.4f} ± {np.std(folds, ddof=1):.4f}  (par fold : {[round(v,4) for v in folds]})")
print(f"Candidats énumérés : {len(thr):,} (valeurs prédites distinctes) — argmax automatique")
print("Indépendance vérifiée : model_final.pkl, calibrator.pkl, decision_threshold.pkl intacts (MD5),")
print("seule entrée du calcul = oof_predictions.csv (phase 3).")

t_m, t_e = main.seuil_surveillance, main.seuil_decision
bandes = {'FAIBLE': p < t_m, 'MOYEN': (p >= t_m) & (p < t_e), 'ELEVE': p >= t_e}
base = y.mean()
print(f"Seuils : surveillance = {t_m:.10f} | décision = {t_e:.10f} (artefacts distincts)")
print(f"\n{'bande':<8} {'part clients':>13} {'taux de défaut observé':>23} {'vs moyenne':>11}")
for nom, m in bandes.items():
    print(f"{nom:<8} {m.mean()*100:>12.1f}% {y[m].mean()*100:>22.2f}% {y[m].mean()/base:>10.1f}x")
rappel = y[p >= t_m].sum() / y.sum()
print(f"\nLa zone surveillée (MOYEN + ÉLEVÉ) capte {rappel*100:.1f}% des défauts futurs "
      f"avec {(p >= t_m).mean()*100:.1f}% des dossiers.")

niveau_moyen_threshold.pkl = 0.1007080147267063
Stabilité inter-folds : 0.1008 ± 0.0104  (par fold : [0.1074, 0.1043, 0.0896, 0.1126, 0.0903])
Candidats énumérés : 465 (valeurs prédites distinctes) — argmax automatique
Indépendance vérifiée : model_final.pkl, calibrator.pkl, decision_threshold.pkl intacts (MD5),
seule entrée du calcul = oof_predictions.csv (phase 3).
Seuils : surveillance = 0.1007080147 | décision = 0.2226562500 (artefacts distincts)

bande     part clients  taux de défaut observé  vs moyenne
FAIBLE           84.2%                   2.72%        0.4x
MOYEN             7.1%                  14.85%        2.2x
ELEVE             8.7%                  38.62%        5.8x

La zone surveillée (MOYEN + ÉLEVÉ) capte 65.8% des défauts futurs avec 15.8% des dossiers.


## 4. Explication du seuil de décision (0.22265625)

**Critère** : maximum du **F1** sur la classe défaut. **Protocole** : prédictions
out-of-fold de la CV 5-fold stratifiée (chaque dossier prédit par un modèle qui ne l'a
jamais vu), calibrées par CV — train uniquement, le test n'a joué aucun rôle dans le choix.
**Sélection automatique** : `precision_recall_curve` énumère TOUTES les valeurs prédites
distinctes comme seuils candidats (465 ici) et l'argmax est pris par programme
(notebook 03, §9) — aucune intervention manuelle. La valeur non ronde en est la preuve :
0.22265625 est une **valeur de sortie du calibreur isotonique** (fonction en escalier),
pas un chiffre choisi. Tout seuil entre deux marches adjacentes (0.22242, 0.22296) donnerait
exactement les mêmes décisions. La phase 4 a validé sa généralisation : F1 test 0.4512,
à 0.0008 de l'optimum a posteriori du test.

In [4]:
from sklearn.metrics import f1_score, precision_score, recall_score, matthews_corrcoef, precision_recall_curve
print('Voisinage du seuil retenu (OOF calibré) — pourquoi pas 0.20, 0.21, 0.23, 0.25 :')
print(f"{'seuil':>9} {'F1':>8} {'précision':>10} {'rappel':>8} {'MCC':>8}")
for t in [0.18, 0.20, 0.21, 0.22, t_e, 0.23, 0.24, 0.25, 0.27]:
    yh = (p >= t).astype(int)
    marq = '  ← argmax F1 (retenu automatiquement)' if t == t_e else ''
    print(f"{t:>9.4f} {f1_score(y, yh):>8.4f} {precision_score(y, yh):>10.4f} "
          f"{recall_score(y, yh):>8.4f} {matthews_corrcoef(y, yh):>8.4f}{marq}")
prec, rec, thr = precision_recall_curve(y, p)
print(f"\nSeuils candidats énumérés automatiquement : {len(thr):,}")
uniq = np.unique(p); i = int(np.searchsorted(uniq, t_e))
print(f"Marches isotoniques adjacentes : {uniq[i-1]:.6f} < {uniq[i]:.6f} (retenu) < {uniq[i+1]:.6f}")
print('Lecture : 0.20/0.21 sacrifient de la précision (plus de fausses alertes), 0.23-0.25')
print('sacrifient du rappel (défauts manqués), pour un F1 inférieur dans les deux directions.')

Voisinage du seuil retenu (OOF calibré) — pourquoi pas 0.20, 0.21, 0.23, 0.25 :
    seuil       F1  précision   rappel      MCC
   0.1800   0.4312     0.3620   0.5332   0.3904


   0.2000   0.4348     0.3765   0.5145   0.3930
   0.2100   0.4351     0.3799   0.5091   0.3930


   0.2200   0.4348     0.3815   0.5054   0.3926


   0.2227   0.4360     0.3862   0.5006   0.3937  ← argmax F1 (retenu automatiquement)
   0.2300   0.4331     0.3938   0.4811   0.3903


   0.2400   0.4333     0.4138   0.4547   0.3910
   0.2500   0.4302     0.4201   0.4409   0.3884


   0.2700   0.4271     0.4331   0.4213   0.3866

Seuils candidats énumérés automatiquement : 465
Marches isotoniques adjacentes : 0.222423 < 0.222656 (retenu) < 0.222956
Lecture : 0.20/0.21 sacrifient de la précision (plus de fausses alertes), 0.23-0.25
sacrifient du rappel (défauts manqués), pour un F1 inférieur dans les deux directions.


## 5. Démarrage en session vierge (application complète)

In [5]:
script = '''
import sys
sys.path.insert(0, sys.argv[1])
import main                                       # démarrage : chargement + validations
from fastapi.testclient import TestClient
c = TestClient(main.app)
h = c.get("/health").json()
assert h["status"] == "ok" and h["seuil_decision"] == 0.22265625
r = c.post("/predict", json={
    "RevolvingUtilizationOfUnsecuredLines": 0.05, "age": 45,
    "NumberOfTime30-59DaysPastDueNotWorse": 0, "DebtRatio": 0.25, "MonthlyIncome": 6500,
    "NumberOfOpenCreditLinesAndLoans": 6, "NumberOfTimes90DaysLate": 0,
    "NumberRealEstateLoansOrLines": 1, "NumberOfTime60-89DaysPastDueNotWorse": 0,
    "NumberOfDependents": 1, "charges_mensuelles": 1625.0, "score_retards": 0.0,
    "historique_financier": 0.0, "nb_credits_total": 7.0}).json()
assert abs(r["probabilite_calibree"] - 0.0089) < 5e-4 and r["niveau_risque"] == "FAIBLE"
assert r["seuil_decision"] == 0.22265625 and abs(r["seuil_surveillance"] - 0.1007080147267063) < 1e-12
print("SESSION VIERGE OK — demarrage + /health + /predict + 2 seuils reproduits, PD =", r["probabilite_calibree"])
'''
r = subprocess.run([sys.executable, '-c', script, os.path.abspath('..')],
                   capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr

SESSION VIERGE OK — demarrage + /health + /predict + 2 seuils reproduits, PD = 0.008864


## 6. Synthèse de déploiement

| Élément | Valeur |
|---|---|
| Point d'entrée | `ml-service/main.py` — `uvicorn main:app --host 0.0.0.0 --port 8000` (URL attendue par le backend : `ia.service.url`, défaut `http://localhost:8000`) |
| Artefacts de production | `model_final.pkl` (ScoringModel : préprocesseur + XGBoost contraint), `calibrator.pkl` (isotonique), `decision_threshold.pkl` (0.22265625, décision), `niveau_moyen_threshold.pkl` (0.1007, watch-list), `feature_cols.pkl`, `metadata_final.pkl` + `src/preprocessing.py` importé avant dépicklage |
| Dépendances service | `requirements.txt` (fastapi, uvicorn, pydantic, numpy, pandas, scikit-learn, xgboost, shap) |
| Dépendances notebooks/tests | `requirements-notebooks.txt` (+ httpx pour les tests) |
| Ordre d'exécution (reproduction) | notebooks 01 → 07 dans l'ordre (02 produit data/processed + preprocessing, 03 les modèles et seuils, 04 la mesure officielle, 05-07 valident) — seed 42, chemins relatifs |
| Résultats officiels (test, phase 4) | ROC-AUC 0.8627 · PR-AUC 0.4060 · KS 0.5726 · Brier 0.0489 · ECE 0.0030 · F1 0.4512 @ 0.2227 |
| Tests | `python tests/test_api_e2e.py` (session vierge, 4 profils, API ≡ notebooks) |

La procédure **full-stack manuelle** (PostgreSQL + Spring Boot + requête + vérification en
base) est documentée dans `docs/phase8_final_validation.md`.

⏸ **STOP — validation utilisateur requise (fin de la phase 8).**